## Data Imports & Common Vairiables

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import plotly.express as px
import datetime

load_dotenv()
ticker = 'TXN'
path_stockdata = os.path.join(os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList'), ticker)
path_analysis_csv = os.path.join(path_stockdata, f'{ticker}--Analysis_DYT-s1v1.csv')

today = datetime.date.today()
cutoff = today.year - 20
cutoff10 = today.year - 10
cutoff5 = today.year - 5
cutoff3 = today.year - 3


## Data Frame Imports

Notable Cleaning
- Convert string dates to datetime
- Filter down to last 20 years of data
- Filter out current year in dvtagr_df as it will have incomplete data
- Convert decimal yields to numeric percentage for easy viewing

In [ ]:
%%capture

dyt_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--DivPrice_History-s1v1.csv'), index_col=0)
dyt_df0['Date'] = pd.to_datetime(dyt_df0['Date'])
dyt_df1 = dyt_df0[dyt_df0['Date'].dt.year >= cutoff]
dyt_df1['FwdDiv%'] = round(dyt_df0['FwdDivYield'] * 100, 2)


dytagr_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Aggregate_Cy_DivPrice_History-s1v1.csv'), index_col=0)
dytagr_df1 = dytagr_df0.query('DateCy >= @cutoff and DateCy < @today.year')
dytagr_df1['Min%'] = round(dytagr_df1['DivYieldMin'] * 100, 2)
dytagr_df1['Max%'] = round(dytagr_df1['DivYieldMax'] * 100, 2)
dytagr_df1['Mean%'] = round(dytagr_df1['DivYieldMean'] * 100, 2)
dytagr_df1['Median%'] = round(dytagr_df1['DivYieldMedian'] * 100, 2)





In [ ]:
print('dyt_df1 Row Count:', len(dyt_df1))
dyt_df1

In [ ]:
print('dytagr_df1 Row Count:', len(dytagr_df1))
dytagr_df1

## View Aggregate Yield Data By year

In [ ]:
dytagr_df1[['Min%', 'Max%', 'Mean%', 'Median%']]

## Calc Mean, Median & Rolling

In [ ]:
mean20 = round(dyt_df1['FwdDiv%'].mean(), 2)
median20 = round(dyt_df1['FwdDiv%'].median(), 2)

dyt_df1_l10 = dyt_df1[dyt_df1['Date'].dt.year >= cutoff10]
mean10 = round(dyt_df1_l10['FwdDiv%'].mean(), 2)
median10 = round(dyt_df1_l10['FwdDiv%'].median(), 2)

dyt_df1_l5 = dyt_df1[dyt_df1['Date'].dt.year >= cutoff5]
mean5 = round(dyt_df1_l5['FwdDiv%'].mean(), 2)
median5 = round(dyt_df1_l5['FwdDiv%'].median(), 2)

dyt_df1_l3 = dyt_df1[dyt_df1['Date'].dt.year >= cutoff3]
mean3 = round(dyt_df1_l3['FwdDiv%'].mean(), 2)
median3 = round(dyt_df1_l3['FwdDiv%'].median(), 2)

print(f'The 20 year mean is: {mean20}%')
print(f'The 20 year median is: {median20}%')
print()
print(f'The 10 year mean is: {mean10}%')
print(f'The 10 year median is: {median10}%')
print()
print(f'The 5 year mean is: {mean5}%')
print(f'The 5 year median is: {median5}%')
print()
print(f'The 3 year mean is: {mean3}%')
print(f'The 3 year median is: {median3}%')
print()

## DYT Gate
- The Gate is the Dividend Yield where the company is considered fair value


In [ ]:
gate = 2.9
gate10 = round((gate * .1) + gate, 2)
gate20 = round((gate * .20) + gate, 2)

print(f'The DYT Gate is: {gate}%')
print(f'The DYT Gate with 10 percent margin of error is: {gate10}%')
print(f'The DYT Gate with 25 percent margin of error is: {gate20}%')

## DYT Plot

In [ ]:
fig1 = px.line(dyt_df1, x='Date', y='FwdDiv%', title='Fwd Yield Per Date')
fig1.update_traces(line_color='blue')
fig1.add_hline(y=gate, line_dash='dot', annotation_text='Gate', annotation_position='bottom left', line_color='red')
fig1.add_hline(y=gate10, line_dash='dot', annotation_text='Gate10', annotation_position='bottom left', line_color='yellow')
fig1.add_hline(y=gate20, line_dash='dot', annotation_text='Gate20', annotation_position='bottom left', line_color='green')
fig1.show()

## Stored Analysis Output

In [ ]:
metrics_json = {
    "type": "dyt",
    "date": today.strftime('%Y-%m-%d'),
    "dyt_data_type": "numeric percentage",
    "dyt_gate": gate,
    "dyt_gate_10%_mos": gate10,
    "dyt_gate_20%_mos": gate20,
}

metrics_json

In [ ]:
metrics_df = pd.DataFrame([metrics_json])
metrics_df

In [ ]:
if os.path.isfile(path_analysis_csv):
    metrics_df.to_csv(path_analysis_csv, mode='a', header=False, index=False)
else:
    metrics_df.to_csv(path_analysis_csv, mode='w', header=True, index=False)

## Notebook End